# Session 7 Lab: Tool Calling and MCP

**Project 4 · Timetable agent + MCP server** · GenAI & Agentic AI Workshop

**Part A:** give Gemini real tools (exam timetable, library hours, room booking, live weather) and write the agent loop yourself.
**Part B:** share the *same* tools with any AI app by turning them into an **MCP server**.

**Before you start:** you need the `GEMINI_API_KEY` secret from Session 3 (🔑 Secrets, Notebook access ON). Run each cell with **Shift + Enter**.

In [ ]:
!pip install -q google-genai mcp

In [ ]:
import datetime, json
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
MODEL = "gemini-2.5-flash"   # any current free model from Google AI Studio works

# Part A: Function calling

## Step 1: Write the tools

Tools are just **normal Python functions**. The college data is fake (same as the Session 6 tool cards); `get_weather` calls a **real** free weather API.

We save them to a file so the MCP server in Part B can reuse them.

In [ ]:
%%writefile campus_tools.py
# campus_tools.py: the tools our agent can use (fake college data + one real weather API)
import datetime
import requests

EXAMS = {
    "DBMS": {"subject": "DBMS", "date": "2026-12-12", "day": "Saturday", "time": "10 AM", "hall": "Hall B"},
    "OS":   {"subject": "OS",   "date": "2026-12-14", "day": "Monday",   "time": "2 PM",  "hall": "Hall A"},
    "CN":   {"subject": "CN",   "date": "2026-12-16", "day": "Wednesday", "time": "10 AM", "hall": "Hall C"},
}


def get_exam_date(subject: str) -> dict:
    """Get the date, day, time and hall of a student's semester exam. subject: DBMS, OS or CN."""
    exam = EXAMS.get(subject.strip().upper())
    if exam is None:
        return {"error": f"No exam found for '{subject}'. Known subjects: DBMS, OS, CN."}
    return exam


def get_library_hours(date: str) -> dict:
    """Get the central library's opening hours for one date. date: YYYY-MM-DD."""
    try:
        d = datetime.date.fromisoformat(date)
    except ValueError:
        return {"error": "Use the date format YYYY-MM-DD, for example 2026-12-13."}
    day = d.strftime("%A")
    if d.weekday() == 6:                      # Sunday
        if d.month in (5, 12):                # exam months
            return {"date": date, "day": day, "open": "10:00", "close": "16:00",
                    "note": "Sunday opening during exam month"}
        return {"date": date, "day": day, "open": None, "close": None, "note": "Closed on Sundays"}
    return {"date": date, "day": day, "open": "08:00", "close": "20:00"}


BOOKINGS = []


def book_study_room(date: str, start: str, hours: int) -> dict:
    """Book a library study room. date: YYYY-MM-DD, start: HH:MM in 24-hour time, hours: 1 to 3."""
    hours = int(hours)                        # the model may send 2.0 instead of 2
    if not 1 <= hours <= 3:
        return {"status": "failed", "reason": "You can book between 1 and 3 hours only."}
    lib = get_library_hours(date)
    if "error" in lib:
        return {"status": "failed", "reason": lib["error"]}
    if lib["open"] is None:
        return {"status": "failed", "reason": f"The library is closed on {lib['day']} {date}."}
    start_h = int(start.split(":")[0])
    if start_h < int(lib["open"][:2]) or start_h + hours > int(lib["close"][:2]):
        return {"status": "failed", "reason": f"The library is only open {lib['open']}-{lib['close']} that day."}
    room = f"S{len(BOOKINGS) + 1}"
    BOOKINGS.append({"room": room, "date": date, "start": start, "hours": hours})
    return {"status": "confirmed", "room": room, "date": date, "start": start, "hours": hours}


def get_weather(city: str) -> dict:
    """Get the current temperature and today's and tomorrow's rain chance for a city (free Open-Meteo API, no key)."""
    geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                       params={"name": city, "count": 1}, timeout=10).json()
    if not geo.get("results"):
        return {"error": f"City '{city}' not found."}
    place = geo["results"][0]
    wx = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": place["latitude"], "longitude": place["longitude"],
        "current": "temperature_2m",
        "daily": "precipitation_probability_max,temperature_2m_max,temperature_2m_min",
        "timezone": "auto", "forecast_days": 2,
    }, timeout=10).json()
    daily = wx["daily"]
    days = [{"date": daily["time"][i],
             "rain_chance_pct": daily["precipitation_probability_max"][i],
             "max_c": daily["temperature_2m_max"][i],
             "min_c": daily["temperature_2m_min"][i]} for i in range(2)]
    return {"city": place["name"], "now_c": wx["current"]["temperature_2m"], "today": days[0], "tomorrow": days[1]}

In [ ]:
from campus_tools import get_exam_date, get_library_hours, book_study_room, get_weather, BOOKINGS

# Test them directly first. No AI involved yet!
print(get_exam_date("OS"))
print(get_library_hours("2026-12-13"))
print(book_study_room("2026-12-13", "10:00", 2))
print(get_weather("Mysuru"))

TOOLS = {f.__name__: f for f in (get_exam_date, get_library_hours, book_study_room, get_weather)}
BOOKINGS.clear()

## Step 2: Describe the tools to the model

The model never sees your Python code. It only sees a **name**, a **description** and the **parameters** (as JSON Schema). Good descriptions = the model picks the right tool.

In [ ]:
declarations = [
    {
        "name": "get_exam_date",
        "description": "Get the date, day, time and hall of a student's semester exam.",
        "parameters": {
            "type": "object",
            "properties": {"subject": {"type": "string", "description": "Subject code: DBMS, OS or CN"}},
            "required": ["subject"],
        },
    },
    {
        "name": "get_library_hours",
        "description": "Get the central library's opening hours for one date.",
        "parameters": {
            "type": "object",
            "properties": {"date": {"type": "string", "description": "Date in YYYY-MM-DD format"}},
            "required": ["date"],
        },
    },
    {
        "name": "book_study_room",
        "description": "Book a library study room. Only call this when the student asks for a booking.",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string", "description": "Date in YYYY-MM-DD format"},
                "start": {"type": "string", "description": "Start time, HH:MM in 24-hour format"},
                "hours": {"type": "integer", "description": "Number of hours, 1 to 3"},
            },
            "required": ["date", "start", "hours"],
        },
    },
    {
        "name": "get_weather",
        "description": "Get the current temperature and today's and tomorrow's rain chance for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g. Mysuru"}},
            "required": ["city"],
        },
    },
]

SYSTEM = f"""You are CampusBuddy, the helpdesk agent for ABC Engineering College.
Today is {datetime.date.today():%A %d %B %Y}.
- Use the tools to look up exams, library hours, room bookings and weather. Never guess a date or time.
- If a tool returns an error or a failure, explain it simply and suggest an alternative.
- Keep answers short and friendly."""

config = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    tools=[types.Tool(function_declarations=declarations)],
    temperature=0,
)

## Step 3: One round, by hand

Ask a question. Instead of text, the model replies with a **function call**: "please run `get_exam_date` with `subject="OS"`". It does **not** run it.

In [ ]:
question = "When is my OS exam?"
resp = client.models.generate_content(model=MODEL, contents=question, config=config)

for call in resp.function_calls or []:
    print("The model asks us to call:", call.name, dict(call.args))

Now **our code** runs the tool and sends the result back. Then the model writes the answer.

In [ ]:
call = resp.function_calls[0]
result = TOOLS[call.name](**call.args)          # WE run the tool
print("Tool result:", result)

contents = [
    types.Content(role="user", parts=[types.Part(text=question)]),
    resp.candidates[0].content,                 # the model's tool request
    types.Content(role="user", parts=[
        types.Part.from_function_response(name=call.name, response={"result": result}),
    ]),
]
final = client.models.generate_content(model=MODEL, contents=contents, config=config)
print("Answer:", final.text)

## Step 4: The agent loop

Wrap that in a loop (think → act → observe), cap the steps, and ask a human before any **action** that changes something (Session 5).

In [ ]:
NEEDS_APPROVAL = {"book_study_room"}

def run_agent(goal, max_steps=6, ask_human=True):
    contents = [types.Content(role="user", parts=[types.Part(text=goal)])]
    for step in range(1, max_steps + 1):                       # always cap the loop
        resp = client.models.generate_content(model=MODEL, contents=contents, config=config)
        if not resp.function_calls:                             # THINK: no tool needed, so done
            return resp.text
        contents.append(resp.candidates[0].content)
        parts = []
        for call in resp.function_calls:                        # ACT
            args = dict(call.args or {})
            print(f"🛠️  step {step}: {call.name}({args})")
            if call.name in NEEDS_APPROVAL and ask_human and input("   Approve? (y/n) ").strip().lower() != "y":
                result = {"status": "cancelled", "reason": "The student did not approve this action."}
            else:
                result = TOOLS[call.name](**args)
            print(f"👀 {result}")                               # OBSERVE
            parts.append(types.Part.from_function_response(name=call.name, response={"result": result}))
        contents.append(types.Content(role="user", parts=parts))
    return "Sorry, I couldn't finish that. Please contact the office."

print(run_agent("When is my OS exam? If the library is open the day before, book me a study room for 2 hours in the morning."))

Watch the 🛠️ and 👀 lines: that's the ReAct loop from Session 6, for real. Now try the tricky goal cards from Session 6:

In [ ]:
goals = [
    "Book me a 4-hour study room the day before my DBMS exam.",      # tool says max 3 hours
    "When is my Maths exam?",                                       # not in the timetable
    "Will it rain in Mysuru tomorrow? Should I carry an umbrella to college?",
    "How many days are there between my first and last exam?",
]
for g in goals:
    print("🎯", g)
    print("✅", run_agent(g, ask_human=False), "\n")

## Step 5: The shortcut (automatic function calling)

The SDK can do the whole loop for you: pass the **Python functions** directly. It reads their docstrings and type hints, calls them, and sends results back automatically.

Handy for quick scripts. But you lose control: no approval step, no logging of each call. That's why we wrote the loop by hand first.

In [ ]:
auto_config = types.GenerateContentConfig(
    system_instruction=SYSTEM,
    tools=[get_exam_date, get_library_hours, get_weather],   # plain Python functions
)
resp = client.models.generate_content(
    model=MODEL, contents="Is the library open on the day before my CN exam, and until when?", config=auto_config,
)
print(resp.text)

# Part B: MCP (Model Context Protocol)

Our tools only work inside **this notebook**. MCP is a standard way to package tools so **any** AI app (Claude Desktop, VS Code, Cursor, your own agents) can use them, without rewriting anything.

## Step 6: Turn the tools into an MCP server

It's just a few lines, because we reuse `campus_tools.py`:

In [ ]:
%%writefile campus_server.py
# campus_server.py: the SAME tools, shared with any AI app through MCP
# Written for the MCP Python SDK v2 (pip install mcp). In v1 tutorials you'll see
# "from mcp.server.fastmcp import FastMCP" instead; it's the same idea, renamed.
from mcp.server.mcpserver import MCPServer
from campus_tools import get_exam_date, get_library_hours, book_study_room, get_weather

mcp = MCPServer("campus")

# Register each Python function as an MCP tool. Its name, docstring and type hints
# become the tool's name, description and input schema.
for fn in (get_exam_date, get_library_hours, book_study_room, get_weather):
    mcp.add_tool(fn)

if __name__ == "__main__":
    mcp.run()        # talks over stdin/stdout ("stdio"), the simplest MCP transport

## Step 7: Connect to it as an MCP client

An MCP **client** starts the server, asks what tools it has, and calls them. Notice we never import `campus_tools` here: everything goes through MCP.

In [ ]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command=sys.executable, args=["campus_server.py"])

async def explore_server():
    async with stdio_client(server, errlog=open("mcp_server.log", "w")) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            listed = await session.list_tools()
            for t in listed.tools:
                print("🧰", t.name, "-", t.description)
            result = await session.call_tool("get_exam_date", {"subject": "OS"})
            print("\nget_exam_date('OS') →", result.content[0].text)

await explore_server()

## Step 8: Gemini + your MCP server

Now the agent gets its tools **from the MCP server** instead of from our hand-written list. We convert each MCP tool's schema into a Gemini declaration, and send every tool call through MCP.

In [ ]:
def clean_schema(schema):
    """Drop JSON Schema keys Gemini doesn't need (like 'title')."""
    if isinstance(schema, dict):
        return {k: clean_schema(v) for k, v in schema.items() if k not in ("title", "additionalProperties")}
    if isinstance(schema, list):
        return [clean_schema(x) for x in schema]
    return schema

async def run_mcp_agent(goal, max_steps=6):
    async with stdio_client(server, errlog=open("mcp_server.log", "w")) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            listed = await session.list_tools()
            mcp_decls = [{"name": t.name, "description": t.description, "parameters": clean_schema(t.input_schema)}
                         for t in listed.tools]          # (called t.inputSchema in MCP SDK v1)
            mcp_config = types.GenerateContentConfig(
                system_instruction=SYSTEM, tools=[types.Tool(function_declarations=mcp_decls)], temperature=0)

            contents = [types.Content(role="user", parts=[types.Part(text=goal)])]
            for step in range(1, max_steps + 1):
                resp = await client.aio.models.generate_content(model=MODEL, contents=contents, config=mcp_config)
                if not resp.function_calls:
                    return resp.text
                contents.append(resp.candidates[0].content)
                parts = []
                for call in resp.function_calls:
                    args = dict(call.args or {})
                    print(f"🔌 step {step}: MCP → {call.name}({args})")
                    result = await session.call_tool(call.name, args)
                    text = result.content[0].text if result.content else ""
                    print(f"👀 {text}")
                    parts.append(types.Part.from_function_response(name=call.name, response={"result": text}))
                contents.append(types.Content(role="user", parts=parts))
            return "Sorry, I couldn't finish that."

print(await run_mcp_agent("Is the library open the evening before my CN exam? What's the weather in Mysuru today?"))

Same answers as Part A, but now the tools live in a separate server that **any** MCP app can use.

## Step 9: Use your server in a real AI app (on your laptop)

Colab can't run desktop apps, so try this on your own laptop: download `campus_tools.py` and `campus_server.py` (Files panel on the left → ⋮ → Download), then `pip install mcp requests`.

**Claude Desktop:** Settings → Developer → Edit Config, and add:

```json
{
  "mcpServers": {
    "campus": {
      "command": "python",
      "args": ["C:/path/to/campus_server.py"]
    }
  }
}
```

**Claude Code** (terminal): `claude mcp add campus -- python /path/to/campus_server.py`

**VS Code / Cursor:** add the same server in the editor's MCP settings.

Restart the app and ask: *"When is my OS exam?"* It will ask permission, then call **your** tool.

## Stretch goals

1. **Add a tool:** add `search_faq(question)` from Session 4 to `campus_tools.py` and register it in the server. Now one agent can answer FAQ questions *and* use live tools.
2. **Better errors:** what happens if you ask for a booking on `"13/12/2026"`? Improve the tool's error message so the model can fix its own mistake.
3. **Remote server:** change `mcp.run()` to `mcp.run(transport="streamable-http")` to serve over HTTP instead of stdio.
4. **Safety:** add a tool `send_email_to_all_students(message)` that only *prints*. Try a prompt injection (Session 5) that makes the agent call it. Does the approval step save you?

## Done? 🎉

Save a copy and push it, with `campus_tools.py` and `campus_server.py`, to your GitHub as Project 4.